In [34]:
import pandas as pd
from pathlib import Path
import yaml

In [35]:
configs = yaml.safe_load(Path("../config.yaml").read_text())


In [36]:
configs

{'chunking': {'chunk_size': 300, 'chunk_overlap': 50},
 'models': {'embedding': 'intfloat/multilingual-e5-large',
  'reranker': 'BAAI/bge-reranker-v2-m3'},
 'search': {'top_k': 10,
  'rerank_candidates': 50,
  'rrf_k': 60,
  'aggregation': 'max'},
 'paths': {'raw': 'data/candidate_public/candidate_data/articles.f',
  'calibration_raw': 'data/candidate_public/candidate_data/calibration.f',
  'test': 'data/candidate_public/candidate_data/test.f',
  'reports': 'reports/reports.md',
  'calibration': 'data/queries.jsonl'}}

In [39]:
df = pd.read_feather("../" + configs["paths"]["raw"])
calibration = pd.read_feather("../" + configs['paths']["calibration_raw"])
test = pd.read_feather("../" + configs["paths"]["test"])

/home/dmitriy/projects/avito-rag-search/.venv/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [40]:
df

,article_id,title,body
0,1730,Имя или название компании,"<ol><li><p>Зайдите в раздел <a href=""https://w..."
1,1746,"Понять, что профиль заблокирован","<p>Проверьте, какое сообщение вы видите при вх..."
2,1747,Не допустить блокировки профиля,<ol><li><p><strong>Не заводите несколько аккау...
3,1774,Оставить или удалить профиль,<p>⚡ Не удаляйте профиль с подтверждёнными дан...
4,1775,Удалить профиль,"<p>⚡ Удалить профиль не получится, если у вас ..."
...,...,...,...
788,4545,Лента постов от продавца,<p>Теперь на Авито можно публиковать не только...
789,4547,Записаться на тест-драйв,<p>Записаться на тест-драйв</p>
790,4548,Автоматический чат с продавцом,"<p>Иногда покупатель интересуется объявлением,..."
791,4549,Автоматический чат с продавцом,"<p>Иногда покупатель интересуется объявлением,..."


In [41]:
df['article_id'].nunique() == df['article_id'].count()

np.True_

In [42]:
calibration

,query_id,query_text,ground_truth
0,1,Как передать товар через службу авито,1909 4234
1,2,"Можете подсказать, если заказать товар Авито д...",2865 4400
2,3,Здравствуйте. Как отправить товар через Авито.,1909
3,4,как получить деньги за возрат если продавец уж...,4400 4403
4,5,"Когда мне прийдут деньги за доставку, сегодня ...",4361
...,...,...,...
495,496,Хочу закрыть кошелек и чтобы деньги за продажи...,4384
496,497,Сколько платит покупатель за доставку?,1951 4234
497,498,Почему нету доставки за <MONEY>?,1951 4234
498,499,Добрый вечер 🌇 Присылали промокод на <MONEY> н...,2665 4214


In [43]:
test

,query_id,query_text
0,1,"Здравствуйте! Подскажите, пожалуйста, не могу ..."
1,2,"Здравствуйте , почему так долго доставляется в..."
2,3,"Здравствуйте,подскажите как мне отправить крос..."
3,4,Здравствуйте! В каких случаях за возврат снима...
4,5,Почему у меня доставки в несколько раз дороже ...
...,...,...
495,496,Отправила заказ авито доставкой . Куда не вижу...
496,497,"Добрый вечер , мой заказ потерян, могу ли я по..."
497,498,Не пришли деньги за товар. Что делать
498,499,Как добавить бесплатную доставку в уже активно...


In [48]:
df['body'].iloc[1]

'<p>Проверьте, какое сообщение вы видите при входе в профиль.</p><div class="tabset tabset_155"><input type="radio" name="tab_radio_155" id="set155_tab1" class="tab-radio" checked><label for="set155_tab1" class="tab-label" data-tab-name="1"><p>Профиль заблокирован</p></label><input type="radio" name="tab_radio_155" id="set155_tab2" class="tab-radio"><label for="set155_tab2" class="tab-label" data-tab-name="2"><p>Доступ к профилю ограничен</p></label><input type="radio" name="tab_radio_155" id="set155_tab3" class="tab-radio"><label for="set155_tab3" class="tab-label" data-tab-name="3"><p>Доступ ограничен. Пройдите проверку по лицу</p></label><div class="tab-panels"><div class="tab-panel"><img src="https://www.avito.ru/files/helpcenter/New%20hc/Profile/blocked/profile_blocked_screen_1.png" alt="Экран «Профиль заблокирован»"><p>Фраза под <strong>Профиль заблокирован</strong> будет отличаться в зависимости от <a href="https://support.avito.ru/articles/3432">причины блокировки</a>.</p><div 